# Corazón de Pulpa · Caras que hablan con SadTalker (imagen + audio)

1. **Entorno de ejecución → Cambiar tipo de entorno → GPU T4 → Guardar.**
2. **Entorno de ejecución → Ejecutar todo.**
3. Cuando salga **Elegir archivos**, suban `entrada.zip`.
4. Al final se descarga `salida.zip`. Pónganlo en `Pipeline/colab/` y avísenle a Claude.

La instalación tarda unos 10 minutos; después, 1 o 2 minutos por clip. Si una celda muestra un error, cópienlo completo y péguenlo en el chat.

In [ ]:
# 1. GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Instalar SadTalker con arreglos para las versiones actuales de Colab
%cd /content
!rm -rf SadTalker && git clone -q https://github.com/OpenTalker/SadTalker.git
%cd /content/SadTalker
!pip -q install face_alignment==1.3.5 yacs pydub kornia gfpgan basicsr facexlib safetensors imageio-ffmpeg librosa
!sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/' $(python -c 'import basicsr,os;print(os.path.dirname(basicsr.__file__))')/data/degradations.py
!grep -rl 'np\.float\b' src | xargs -r sed -i 's/np\.float\b/float/g'
!sed -i 's/np.array(\[w0, h0, s, t\[0\], t\[1\]\])/np.array([w0, h0, s, float(t[0]), float(t[1])])/' src/face3d/util/preprocess.py
!sed -i 's/warnings.filterwarnings("ignore", category=np.VisibleDeprecationWarning)//' src/face3d/util/preprocess.py
!sed -i 's/librosa.filters.mel(hp.sample_rate, hp.n_fft,/librosa.filters.mel(sr=hp.sample_rate, n_fft=hp.n_fft,/' src/utils/audio.py
!bash scripts/download_models.sh > /dev/null 2>&1; ls checkpoints gfpgan/weights

In [ ]:
# Subir entrada.zip
from google.colab import files
import zipfile, os, shutil
shutil.rmtree('/content/entrada', ignore_errors=True)
up = files.upload()
for n in up: zipfile.ZipFile(n).extractall('/content/entrada')
print(sorted(os.listdir('/content/entrada')))

In [ ]:
# 4. Animar: --still y --preprocess full conservan la imagen completa; expression_scale da más gesto
import glob, subprocess, os, shutil
os.makedirs('/content/salida', exist_ok=True)
for img in sorted(glob.glob('/content/entrada/*.png')):
    fila = os.path.basename(img)[:-4]; wav = f'/content/entrada/{fila}.wav'
    if not os.path.exists(wav): print('sin audio:', fila); continue
    for modo, extra in (('still', ['--still']), ('vivo', [])):     # dos versiones: cabeza quieta y cabeza con más movimiento
        out = f'/content/tmp/{fila}-{modo}'; shutil.rmtree(out, ignore_errors=True)
        print('animando', fila, modo)
        r = subprocess.run(['python', 'inference.py', '--driven_audio', wav, '--source_image', img, '--result_dir', out,
                            '--preprocess', 'full', '--enhancer', 'gfpgan', '--expression_scale', '1.3'] + extra,
                           capture_output=True, text=True)
        vids = glob.glob(out + '/**/*.mp4', recursive=True)
        if vids: shutil.copy(max(vids, key=os.path.getsize), f'/content/salida/{fila}-{modo}.mp4'); print('  listo')
        else: print('  ERROR\n', r.stdout[-1500:], r.stderr[-3000:])

In [ ]:
# Descargar salida.zip
!cd /content/salida && zip -q -r /content/salida.zip . && ls -la /content/salida
from google.colab import files
files.download('/content/salida.zip')